# PPE zero-shot VLM evaluation — no Pandas needed

This edition avoids **Pandas, NumPy, Matplotlib and scikit-learn**, which can require native DLL files blocked by some Windows Application Control settings. It relies on Python's standard libraries for CSV processing and metrics; the Ollama client and Jupyter/IPython are still required for model requests and notebook display.

- Compare your **human-verified image labels** to VLM predictions (zero-shot; **no training**).
- Measure **accuracy, precision, recall, F1, false-positive rate, prediction coverage, strict accuracy**, and processing time.
- Output one CSV per model plus a reusable `model_comparison.csv`.
- Display a text confusion matrix, so you don't need Matplotlib.

**Scope:** image-level compliance means **at least one visible worker clearly lacks a helmet or high-visibility vest**. The prompt asks about boots, but boots are *not scored* unless you have verified ground truth for boots. The model must not infer certified safety footwear just from appearance.


## 1. Install Ollama client (only if needed)


In [28]:
%pip install ollama

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## 2. Settings

Change `IMAGE_DIR` to the folder with your PPE images and `LABELS_CSV` to its ground-truth CSV. `RUN_LIMIT = 5` checks five **verified** images first. Set `0` to evaluate all verified images.

Use `MODE = "cloud"` only when your selected model is available through a configured hosted Ollama provider. Cloud processing sends your images to that provider; use only approved test images.


In [ ]:
from pathlib import Path
from getpass import getpass
from ollama import Client
import csv
import json
import re
import time
from IPython.display import display

# -------------------------------------------------------------------------------------------------
# Choose local, remote or cloud model. Ctrl + / the model + mode you DON'T want to use.

MODEL_CHOICE = "minicpm-v4.6:1b"
MODE = "local"
# openbmb/minicpm-v4.7:1b, minicpm-v4.6:1b, gemma3:4b, llava:7b, qwen3-vl:8b (144s per image, takes forever to run)

# MODEL_CHOICE = "qwen3.8:27b" # currently only remote model
# MODE = "remote"                 

# --------------------------------------------------------------------------------------------------

RUN_LIMIT = 0                 # 0 = evaluate all verified images
REQUEST_TIMEOUT = 300          # seconds per request
SHOW_IMAGES = False            # True = display test images during processing

IMAGE_DIR = Path(r"C:\Users\GYS\Downloads\test model\model testing images")
LABELS_CSV = Path("ground_truth_review.csv")

# Example balanced dataset (adjust to your extracted location):
# DATASET_DIR = Path(r"C:\Users\GYS\Downloads\PPE_Balanced_VLM_Evaluation")
# IMAGE_DIR = DATASET_DIR / "test" / "images"
# LABELS_CSV = DATASET_DIR / "ground_truth_review.csv"

RESULTS_DIR = Path.cwd() / "ppe_model_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


if MODE == "remote":
    client = Client(
        host="http://172.16.167.18:80",
        timeout=REQUEST_TIMEOUT
    )

elif MODE == "cloud":
    api_key = getpass("Ollama Cloud API key: ")
    client = Client(
        host="https://ollama.com",
        headers={"Authorization": f"Bearer {api_key}"},
        timeout=REQUEST_TIMEOUT
    )

else:
    client = Client(
        host="http://localhost:11434",
        timeout=REQUEST_TIMEOUT
    )

print(f"Model: {MODEL_CHOICE} | Mode: {MODE}")
print(f"Images: {IMAGE_DIR}")
print(f"Labels: {LABELS_CSV}")


Model: minicpm-v4.6:1b | Mode: local
Images: C:\Users\GYS\Downloads\test model\model testing images
Labels: ground_truth_review.csv


## 3. Prepare and verify ground truth

- `verified_helmet` and `verified_vest` use **yes** if one or more visible workers clearly lack that item, **no** if every assessable worker wears it, and **unclear** if you can't tell and no missing PPE is certain.
- `review_status` must be **verified** for a row to be scored.
- A dataset owner's annotation is a starting point; visually verify it. No missing annotation does **not** prove compliance.

If no CSV exists, this cell makes a **blank** review CSV. Open it in Excel, fill in the labels, and save it as CSV. It can also read the earlier review format with `verified_label` = `violation` / `compliant` / `unclear`.


In [30]:
from IPython.display import Image as NotebookImage

if not IMAGE_DIR.is_dir():
    raise FileNotFoundError(f"Image folder not found: {IMAGE_DIR}. Edit IMAGE_DIR in Settings.")

all_images = sorted(p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
if not all_images:
    raise ValueError(f"No JPG/PNG images found in: {IMAGE_DIR}")

if not LABELS_CSV.exists():
    with LABELS_CSV.open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=['image', 'verified_helmet', 'verified_vest', 'review_status', 'notes'])
        writer.writeheader()
        writer.writerows({'image': p.name, 'verified_helmet': '', 'verified_vest': '',
                          'review_status': 'needs_review', 'notes': ''} for p in all_images)
    print(f"Blank review CSV created at: {LABELS_CSV}")
    print("Open this file in Excel, check your images and complete the labels first.")

with LABELS_CSV.open(newline='', encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    headers = set(reader.fieldnames or [])
    labels = list(reader)

if not {'image', 'review_status'} <= headers:
    raise ValueError("Ground-truth CSV needs 'image' and 'review_status' columns.")

if {'verified_helmet', 'verified_vest'} <= headers:
    data_type = 'per_item'
    allowed = {'yes', 'no', 'unclear'}
    selected_labels = [r for r in labels
        if r['review_status'].strip().lower() == 'verified'
        and str(r.get('verified_helmet','')).strip().lower() in allowed
        and str(r.get('verified_vest','')).strip().lower() in allowed]
elif 'verified_label' in headers:
    data_type = 'overall_only'
    selected_labels = [r for r in labels
        if r['review_status'].strip().lower() == 'verified'
        and str(r.get('verified_label','')).strip().lower() in {'violation','compliant','unclear'}]
else:
    raise ValueError("CSV needs either verified_helmet + verified_vest, or verified_label.")

for row in selected_labels:
    for field in ('verified_helmet', 'verified_vest', 'verified_label'):
        if field in row:
            row[field] = str(row[field]).strip().lower()
selected_labels = [r for r in selected_labels if (IMAGE_DIR / r['image']).is_file()]

print(f"Found {len(all_images)} images, {len(labels)} CSV entries, {len(selected_labels)} verified images.")
print("Ground-truth format:", data_type)
if not selected_labels:
    print("No verified images yet. Check the images, update the CSV and rerun this cell.")
else:
    for row in selected_labels[:6]:
        print(' ', row['image'], '|', row.get('verified_helmet',''), row.get('verified_vest',''), row.get('verified_label',''))


Found 60 images, 60 CSV entries, 60 verified images.
Ground-truth format: per_item
  000005_jpg.rf.c613c066b79795d219855aca269760e6.jpg | no yes 
  006672_jpg.rf.657b0a9ecd720d07ca5dd75338a058d5.jpg | no no 
  002551_jpg.rf.b09484b130c1587abfa8365c3d774bf1.jpg | no yes 
  IMG_5846_jpg.rf.f82d7ebbe6b6c050cacb8a7e0ff64424.jpg | no no 
  003357_jpg.rf.9fb1f2cebc239b4276a81ce2cbf9e911.jpg | no yes 
  image_53_jpg.rf.126c05efe93729b667ca2860e6fd0ac3.jpg | no no 


## 4. Prompt and parsing helpers

For fair comparisons, use exactly the same image and prompt for every model. The prompt requests JSON so Python can measure *actual predictions* rather than guess from free-form text. The script considers an invalid response an `error`, not a compliant image.

In [31]:
PROMPT = """Assess this construction-site image for PPE compliance.
Inspect every visible worker for a safety helmet and a high-visibility vest.
Also describe visible safety boots, but do NOT infer certified safety footwear from appearance alone.

For helmet/vest classifications: yes = at least one worker is clearly missing it;
no = all visible workers can be assessed and none are missing it;
unclear = at least one worker cannot be assessed, and no definite absence is visible.
Never guess when the relevant body part is obstructed.

Return ONLY valid JSON, with exactly these fields and permitted values:
{
  "worker_count": 2,
  "workers": [
    {"position": "left", "helmet": "worn", "vest": "missing", "boots": "unclear"}
  ],
  "helmet_violation": "no",
  "vest_violation": "yes",
  "boots_violation": "unclear"
}
For every violation field, choose only one of: yes, no, unclear. The JSON above is an example of formatting, NOT the expected answer.
"""

def overall_from_items(helmet, vest):
    """Image-level decision from helmet/vest only (boots not in ground truth)."""
    if 'yes' in (helmet, vest):
        return 'violation'
    if helmet == 'no' and vest == 'no':
        return 'compliant'
    return 'unclear'

def parse_vlm_response(raw):
    """Parse valid JSON; explicitly report noncompliant/unparseable responses."""
    raw = str(raw or '').strip()
    raw = re.sub(r'^```(?:json)?\s*|\s*```$', '', raw, flags=re.I)
    try:
        obj = json.loads(raw)
    except (ValueError, TypeError):
        left, right = raw.find('{'), raw.rfind('}')
        if left < 0 or right <= left:
            return None
        try:
            obj = json.loads(raw[left:right + 1])
        except (ValueError, TypeError):
            return None
    if not isinstance(obj, dict):
        return None
    for field in ('helmet_violation', 'vest_violation'):
        v = obj.get(field)
        if not isinstance(v, str) or v.strip().lower() not in ('yes', 'no', 'unclear'):
            return None
        obj[field] = v.strip().lower()
    boots = str(obj.get('boots_violation', 'unclear')).strip().lower()
    obj['boots_violation'] = boots if boots in ('yes','no','unclear') else 'unclear'
    return obj

def truth_for_row(row):
    if data_type == 'per_item':
        h, v = row['verified_helmet'], row['verified_vest']
        return h, v, overall_from_items(h, v)
    overall = row['verified_label']
    return '', '', overall

# Small offline sanity checks: no Ollama call or model download.
assert overall_from_items('no','no') == 'compliant'
assert overall_from_items('yes','unclear') == 'violation'
assert overall_from_items('no','unclear') == 'unclear'
assert parse_vlm_response('{"helmet_violation":"yes","vest_violation":"no"}') is not None
print('Parsing and ground-truth helpers are ready.')

Parsing and ground-truth helpers are ready.


## 5. Analyse verified images with the selected model

Each image is analysed separately. Model failures or non-JSON responses are marked as `error`—not silently interpreted as compliant. We time every request including any network latency.


In [32]:
if not selected_labels:
    raise ValueError("No verified labels yet. Complete ground_truth_review.csv and rerun section 3.")

run_rows = selected_labels[:RUN_LIMIT] if RUN_LIMIT > 0 else selected_labels
predictions = []
run_started = time.perf_counter()

for i, row in enumerate(run_rows, start=1):
    path = IMAGE_DIR / row['image']
    truth_h, truth_v, truth_overall = truth_for_row(row)
    print(f"[{i}/{len(run_rows)}] {path.name}")
    if SHOW_IMAGES:
        display(NotebookImage(filename=str(path), width=320))

    started = time.perf_counter()
    raw, parsed, error = '', None, ''
    try:
        response = client.chat(
            model=MODEL_CHOICE,
            messages=[{'role':'user','content':PROMPT,'images':[path.read_bytes()]}],
            format='json',
            options={'temperature': 0}
        )
        raw = response.message.content or ''
        parsed = parse_vlm_response(raw)
        if parsed is None:
            error = 'Unparseable JSON or invalid helmet/vest classification'
    except Exception as exc:
        error = f'{type(exc).__name__}: {exc}'

    seconds = time.perf_counter() - started
    predicted_h = parsed['helmet_violation'] if parsed else 'error'
    predicted_v = parsed['vest_violation'] if parsed else 'error'
    predicted_o = overall_from_items(predicted_h, predicted_v) if parsed else 'error'
    predictions.append({
        'image': path.name,
        'model': MODEL_CHOICE,
        'ground_truth': truth_overall,
        'predicted': predicted_o,
        'truth_helmet': truth_h,
        'pred_helmet': predicted_h,
        'truth_vest': truth_v,
        'pred_vest': predicted_v,
        'pred_boots': parsed['boots_violation'] if parsed else 'error',
        'pred_worker_count': parsed.get('worker_count', '') if parsed else '',
        'seconds': round(seconds, 3),
        'error': error,
        'raw_response': raw,
    })
    print(f"  Ground truth: {truth_overall} | Prediction: {predicted_o} | {seconds:.2f}s")
    if error:
        print(f"  ERROR: {error}")

total_seconds = time.perf_counter() - run_started
print(f"\nAll images processed! {len(predictions)} images | {total_seconds:.2f}s total wall time")
for r in predictions[:10]:
    print(f"  {r['image'][:35]:35} truth={r['ground_truth']:10} pred={r['predicted']:10} time={r['seconds']:.2f}s")


[1/60] 000005_jpg.rf.c613c066b79795d219855aca269760e6.jpg
  Ground truth: violation | Prediction: compliant | 12.55s
[2/60] 006672_jpg.rf.657b0a9ecd720d07ca5dd75338a058d5.jpg
  Ground truth: compliant | Prediction: compliant | 7.89s
[3/60] 002551_jpg.rf.b09484b130c1587abfa8365c3d774bf1.jpg
  Ground truth: violation | Prediction: violation | 8.16s
[4/60] IMG_5846_jpg.rf.f82d7ebbe6b6c050cacb8a7e0ff64424.jpg
  Ground truth: compliant | Prediction: compliant | 6.08s
[5/60] 003357_jpg.rf.9fb1f2cebc239b4276a81ce2cbf9e911.jpg
  Ground truth: violation | Prediction: violation | 7.24s
[6/60] image_53_jpg.rf.126c05efe93729b667ca2860e6fd0ac3.jpg
  Ground truth: compliant | Prediction: compliant | 7.41s
[7/60] 004063_jpg.rf.6ecc587c20a87ec176841aea39b8f5db.jpg
  Ground truth: violation | Prediction: violation | 11.13s
[8/60] image_55_jpg.rf.51b5b121599680f79fff1ad2d6aefba0.jpg
  Ground truth: compliant | Prediction: compliant | 11.01s
[9/60] 004763_jpg.rf.c8f1bd1cb7cf6037f7c90a6c213d6364.jpg
  Gro

## 5A Analyse response reliability (errors vs uncertain classifications)
A parseable unclear response is not a request failure. A model can run successfully but produce unusable JSON; we count those as errors.

In [38]:
# A request can complete while the returned content fails JSON validation.
# "Decisive" means the model returned a usable binary compliance prediction.
from collections import Counter

outcomes = Counter(row['predicted'] for row in predictions)
image_count = len(predictions)
decisive_count = outcomes['violation'] + outcomes['compliant']
unclear_count = outcomes['unclear']
error_count = outcomes['error']
valid_response_count = image_count - error_count

print(f"Images attempted:                 {image_count}")
print(f"Valid, parseable classifications: {valid_response_count}")
print(f"  Decisive (violation/compliant):  {decisive_count}")
print(f"  Unclear assessments:            {unclear_count}")
print(f"Unsuccessful (error) analyses:   {error_count}")
empty_response_errors = sum(
    r['predicted'] == 'error'
    and r['error'].startswith('Unparseable JSON')
    and not (r['raw_response'] or '').strip()
    for r in predictions
)
print(f"  Empty model responses:         {empty_response_errors}")
print(f"  Other format/request errors:   {error_count - empty_response_errors}")

if error_count:
    print("\nFiles requiring error review:")
    for r in predictions:
        if r['predicted'] == 'error':
            print(f"- {r['image']}: {r['error']}")

# These are the counts of model outputs, NOT the accuracy scores.


Images attempted:                 60
Valid, parseable classifications: 58
  Decisive (violation/compliant):  57
  Unclear assessments:            1
Unsuccessful (error) analyses:   2
  Empty model responses:         1
  Other format/request errors:   1

Files requiring error review:
- youtube-617_jpg.rf.0f5ea7ba0ae177b6428e831987a50f8b.jpg: Unparseable JSON or invalid helmet/vest classification
- thumbnail-ba5c72edb320b49a69e86b05775c49b2-scaled-1_jpeg_jpg.rf.edefd0830c6b64605136df7a135fdc4f.jpg: Unparseable JSON or invalid helmet/vest classification


troubleshooting

In [37]:
# if prediction had error this will have output of the image name, error message and the raw response from the model
for result in predictions:
    if result["predicted"] == "error":
        print("Image:", result["image"])
        print("Error:", result["error"])
        print("\nOriginal model response:")
        print(repr(result["raw_response"]))

Image: youtube-617_jpg.rf.0f5ea7ba0ae177b6428e831987a50f8b.jpg
Error: Unparseable JSON or invalid helmet/vest classification

Original model response:
''
Image: thumbnail-ba5c72edb320b49a69e86b05775c49b2-scaled-1_jpeg_jpg.rf.edefd0830c6b64605136df7a135fdc4f.jpg
Error: Unparseable JSON or invalid helmet/vest classification

Original model response:
'{\n  "worker_count": 8,\n  "workers": [\n    {"position": "leftmost", "helmet": "no", "vest": "no", "boots": "unclear"},\n    {"position": "next", "helmet": "no", "vest": "no", "boots": "unclear"},\n    {"position": "middle with pink coat", "helmet": "no", "vest": "no", "boots": "unclear"},\n    {"position": "woman in cream sweater", "helmet": "no", "vest": "no", "boots": "white shoes (not safety boots)},",\n    "{\\"position\\": \\"man in dark jacket\\", \\"helmet\\": \\"no\\", \\"vest\\": \\"no\\", \\"boots\\": \\"not visible\\"},\\"position\\": \\"other\\", \\"helmet\\": \\"no\\", \\"vest\\": \\"no\\", \\"boots\\": \\"unclear\\"}":"",\n  

## 6. Evaluate accuracy, precision, recall, F1 and the confusion matrix

Scores are computed on **binary, decisive** predictions, with separate measures for cases where the model responds `unclear` or fails:

- **Coverage:** fraction of verified binary images yielding a decisive `violation` or `compliant` result.
- **Strict accuracy:** correct predictions divided by *all* verified binary images (unclear/errors count as not correct).
- **Actionable recall:** correctly detected violations divided by *all* confirmed violations (unclear/errors count as missed for this measure).

A class absent from your test set makes some metrics **undefined**, not zero. The text confusion matrix uses the usual values TN, FP, FN, TP.


In [34]:
def safe_div(numerator, denominator):
    return numerator / denominator if denominator else None

def metric_report(rows, truth_col, prediction_col, positive='violation', negative='compliant'):
    labeled = [r for r in rows if r[truth_col] in (positive, negative)]
    decided = [r for r in labeled if r[prediction_col] in (positive, negative)]
    tp = sum(r[truth_col] == positive and r[prediction_col] == positive for r in decided)
    tn = sum(r[truth_col] == negative and r[prediction_col] == negative for r in decided)
    fp = sum(r[truth_col] == negative and r[prediction_col] == positive for r in decided)
    fn = sum(r[truth_col] == positive and r[prediction_col] == negative for r in decided)
    total_positive = sum(r[truth_col] == positive for r in labeled)
    return {
        'ground_truth_binary': len(labeled),
        'scored_decisive': len(decided),
        'unclear_or_error_on_binary': len(labeled) - len(decided),
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
        'accuracy': safe_div(tp + tn, len(decided)),
        'precision': safe_div(tp, tp + fp),
        'recall': safe_div(tp, tp + fn),
        'f1': safe_div(2 * tp, 2 * tp + fp + fn),
        'false_positive_rate': safe_div(fp, fp + tn),
        'coverage': safe_div(len(decided), len(labeled)),
        'strict_accuracy': safe_div(tp + tn, len(labeled)),
        'actionable_recall': safe_div(tp, total_positive),
    }

# Unit checks using synthetic outcomes, NOT actual PPE model scores
_test = [
    {'gold':'violation','pred':'violation'},
    {'gold':'compliant','pred':'violation'},
    {'gold':'violation','pred':'unclear'},
    {'gold':'compliant','pred':'compliant'}
]
_check = metric_report(_test, 'gold', 'pred')
assert (_check['tp'], _check['tn'], _check['fp'], _check['fn']) == (1,1,1,0)
assert abs(_check['strict_accuracy'] - 0.5) < 1e-9
assert abs(_check['actionable_recall'] - 0.5) < 1e-9

reports = {'overall': metric_report(predictions, 'ground_truth', 'predicted')}
if data_type == 'per_item':
    reports['helmet'] = metric_report(predictions, 'truth_helmet', 'pred_helmet', 'yes', 'no')
    reports['vest'] = metric_report(predictions, 'truth_vest', 'pred_vest', 'yes', 'no')

average_seconds = safe_div(sum(r['seconds'] for r in predictions), len(predictions))
print(f"Model: {MODEL_CHOICE} | Images: {len(predictions)}")
print(f"Average seconds/image: {average_seconds:.2f}" if average_seconds is not None else "No timing data")
print(f"Total model-request seconds: {sum(r['seconds'] for r in predictions):.2f}")
for scope, report in reports.items():
    print(f"\n--- {scope.upper()} ---")
    for key in ('ground_truth_binary','scored_decisive','tp','tn','fp','fn','accuracy','precision','recall','f1',
                'false_positive_rate','coverage','strict_accuracy','actionable_recall'):
        val = report[key]
        shown = 'undefined' if val is None else f'{val:.3f}' if isinstance(val, float) else val
        print(f'{key:27} {shown}')

r = reports['overall']
print('\nCONFUSION MATRIX (only decisive cases, overall)')
print('                         Predicted compliant   Predicted violation')
print(f"Actual compliant         {r['tn']:>19}   {r['fp']:>19}")
print(f"Actual violation         {r['fn']:>19}   {r['tp']:>19}")


Model: minicpm-v4.6:1b | Images: 60
Average seconds/image: 7.88
Total model-request seconds: 472.71

--- OVERALL ---
ground_truth_binary         60
scored_decisive             57
tp                          16
tn                          30
fp                          0
fn                          11
accuracy                    0.807
precision                   1.000
recall                      0.593
f1                          0.744
false_positive_rate         0.000
coverage                    0.950
strict_accuracy             0.767
actionable_recall           0.552

--- HELMET ---
ground_truth_binary         59
scored_decisive             56
tp                          9
tn                          39
fp                          0
fn                          8
accuracy                    0.857
precision                   1.000
recall                      0.529
f1                          0.692
false_positive_rate         0.000
coverage                    0.949
strict_accuracy        

## 7. Export predictions, metrics and model comparisons

Exports standard CSV files that Excel can open. A model's comparison rows are replaced when you rerun it; other models' rows are retained. No Pandas needed.


In [35]:
def write_csv(path, rows, columns):
    with path.open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=columns, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)

safe_name = re.sub(r'[^a-zA-Z0-9_-]+', '_', MODEL_CHOICE)
results_path = RESULTS_DIR / f'predictions_{safe_name}.csv'
metrics_path = RESULTS_DIR / f'metrics_{safe_name}.csv'
comparison_path = RESULTS_DIR / 'model_comparison.csv'

prediction_columns = list(predictions[0].keys()) if predictions else []
metric_columns = list(next(iter(reports.values())).keys())
write_csv(results_path, predictions, prediction_columns)
write_csv(metrics_path, [{'scope': scope, **m} for scope, m in reports.items()], ['scope'] + metric_columns)

comparison_columns = ['model','scope','images_analyzed','avg_seconds','total_request_seconds'] + metric_columns
new_rows = [{
    'model': MODEL_CHOICE,
    'scope': scope,
    'images_analyzed': len(predictions),
    'avg_seconds': round(average_seconds, 3) if average_seconds is not None else '',
    'total_request_seconds': round(sum(r['seconds'] for r in predictions), 3),
    **m,
} for scope, m in reports.items()]

existing = []
if comparison_path.exists():
    with comparison_path.open(newline='', encoding='utf-8-sig') as f:
        existing = list(csv.DictReader(f))
existing = [r for r in existing if not (r.get('model') == MODEL_CHOICE and r.get('scope') in reports)]
comparison = existing + new_rows
write_csv(comparison_path, comparison, comparison_columns)

print('Saved:', results_path)
print('Saved:', metrics_path)
print('Updated:', comparison_path)
print('\nModel comparison:')
for r in comparison:
    print(f"  {r['model']:28} {r['scope']:8} accuracy={r['accuracy']} recall={r['recall']} f1={r['f1']} avg_seconds={r['avg_seconds']}")


Saved: c:\Users\GYS\Downloads\test model\ppe_model_results\predictions_minicpm-v4_6_1b.csv
Saved: c:\Users\GYS\Downloads\test model\ppe_model_results\metrics_minicpm-v4_6_1b.csv
Updated: c:\Users\GYS\Downloads\test model\ppe_model_results\model_comparison.csv

Model comparison:
  qwen3.8:27b                  overall  accuracy=0.9830508474576272 recall=0.9642857142857143 f1=0.9818181818181818 avg_seconds=17.067
  qwen3.8:27b                  helmet   accuracy=0.9642857142857143 recall=0.9411764705882353 f1=0.9411764705882353 avg_seconds=17.067
  qwen3.8:27b                  vest     accuracy=1.0 recall=1.0 f1=1.0 avg_seconds=17.067
  gemma3:4b                    overall  accuracy=0.55 recall=1.0 f1=0.6823529411764706 avg_seconds=2.707
  gemma3:4b                    helmet   accuracy=0.6779661016949152 recall=0.0 f1=0.0 avg_seconds=2.707
  gemma3:4b                    vest     accuracy=0.5166666666666667 recall=1.0 f1=0.6506024096385542 avg_seconds=2.707
  llava:7b                     ov

### Interpreting your results

- If your sample contains only violations, it cannot measure the false-positive rate. Include genuine compliant images.
- If there are no manually verified labels, accuracy cannot be calculated responsibly.
- `unclear` and invalid responses reduce coverage and strict accuracy even if decisive-only accuracy is high.
- The first model request may be slower because Ollama needs to load the weights into memory; keep the same conditions across tests.
- For all commercial/remote cloud use, ensure your image data is permitted to leave your computer.
- This notebook scores **helmet and vest** only. Boots require separate verified boot labels.


# Connecting qwen3.8:27b


In [36]:

from ollama import Client

# Connect to the remote Ollama server
client = Client(
    host="http://172.16.167.18:80",
    timeout=180
)

# List the models installed on the remote server
models = client.list()

for model in models.models:
    print(model.model)


llava:13b
dlasher/MiniCPM-V-4.7-35B-A3B:IQ4_NL
qwen3.8:27b
all-minilm:33m
qwen3-embedding:latest
gemma4:26b
gpt-oss:20b
mxbai-embed-large:latest
gemma4:e4b
